# Collateral-Aware SOFR Engine Demo

This notebook is structured as a short implementation note rather than a dashboard. The goal is to explain the model in the same spirit as the source papers and then show exactly how the codebase turns that theory into a working curve, pricing, and risk engine.

Primary references used to frame the notebook:

- `sofr_mercurio_2.pdf`: Fabio Mercurio, *A Simple Multi-Curve Model for Pricing SOFR Futures and Other Derivatives*.
- `2010_mar_funding piterbarg (2019_10_05 14_29_35 UTC).pdf`: Vladimir Piterbarg, *Funding beyond discounting: collateral agreements and derivatives pricing*.

In this project, Mercurio provides the SOFR futures and multi-curve modeling backbone, while Piterbarg provides the collateral-aware discounting interpretation.

## 1. Research Question and Paper-Style Roadmap

We want to answer the following practical desk question:

> How do we build a collateral-aware SOFR engine that strips a SOFR projection curve from futures, overlays Hull-White convexity adjustments, prices fixed-vs-SOFR swaps, and reports meaningful curve and scenario risk?

The notebook follows that question in the same order a paper or implementation note would:

1. Define the modeling assumptions and notation.
2. Map the theoretical quantities to the repo modules.
3. Load the market snapshot and conventions.
4. Build the collateral discount curve.
5. Strip the SOFR projection curve from 1m and 3m futures with convexity adjustments.
6. Anchor the longer end with SOFR swaps.
7. Price a collateral-aware SOFR swap.
8. Produce risk and scenario diagnostics.

The bundled dataset is synthetic by design so the entire workflow is reproducible and internally consistent.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'multi_curve_sofr').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from multi_curve_sofr.bootstrap import (
    bootstrap_from_1m_futures,
    bootstrap_from_3m_futures,
    bootstrap_sofr_short_end,
    build_full_curves,
    load_market_data,
)
from multi_curve_sofr.daycount import yearfrac
from multi_curve_sofr.hw_model import U_j_const_sigma, convexity_1m
from multi_curve_sofr.instruments import build_periods
from multi_curve_sofr.pricers import par_swap_rate, pv_swap
from multi_curve_sofr.risk import key_rate_dv01, pv01, run_scenarios

plt.style.use('ggplot')
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

market = load_market_data(PROJECT_ROOT)
result = build_full_curves(PROJECT_ROOT)
result_no_convexity = build_full_curves(PROJECT_ROOT, sigma_override=0.0)

valuation_date = market.config.market.valuation_date
mean_reversion = market.config.model.mean_reversion
sigma = result.sigma

print(f'Valuation date: {valuation_date}')
print(f'Mean reversion a: {mean_reversion:.4f}')
print(f'Convexity sigma: {sigma:.4f}')
print(f'Synthetic data default: {market.config.market.synthetic_data_default}')

## 2. Model Setup and Notation

Following the spirit of Mercurio's multi-curve setup, we work with separate objects for discounting and SOFR projection.

We use a one-factor Hull-White overlay of the form

$$
r_d(t) = x(t) + \alpha_d(t),
$$

$$
s(t) = x(t) + \beta_s(t),
$$

$$
dx(t) = -a\,x(t)\,dt + \sigma\,dW_t,
$$

where:

- $r_d(t)$ is the collateral discounting short rate,
- $s(t)$ is the SOFR-related short rate used for projection,
- $x(t)$ is the common Gaussian factor,
- $a$ is mean reversion,
- $\sigma$ is the constant volatility used in the convexity layer.

In the no-OIS-option-data configuration, the repo fixes $a$ from `conventions.yaml` and calibrates the constant $\sigma$ by minimizing a SOFR projection-curve roughness objective. The objective is the mean squared curvature of adjacent log-discount-implied forwards after bootstrapping futures and swaps. This implements the smoothness alternative described by Mercurio when option data are unavailable.

This yields two key curve concepts:

- $P_d(0,T)$: collateral discount factors used for PVs.
- $P_s(0,T)$: SOFR pseudo-discount factors used to infer forward SOFR rates.

Under the deterministic-basis interpretation, forward SOFR rates are obtained from the SOFR pseudo-discount curve by

$$
F^s_i(0) = \frac{1}{\tau_i}\left(\frac{P_s(0,T_{i-1})}{P_s(0,T_i)} - 1\right).
$$

This is the basic projection identity used throughout the repo.

## 3. Collateral-Aware Valuation Interpretation

Piterbarg's main message is that collateral terms affect discounting and therefore valuation itself. In a stylized fully collateralized setup, the value of a payoff $X_T$ is naturally written as a collateral-discounted expectation:

$$
V_t = \mathbb{E}^{\mathbb{Q}}_t\left[e^{-\int_t^T c(u)\,du} X_T\right],
$$

where $c(u)$ is the collateral remuneration rate. In this notebook:

- the OIS curve is the proxy for the collateral discount curve,
- the SOFR curve is the projection curve for floating cashflows,
- the simple funding-spread scenarios later in the notebook are didactic overlays inspired by the same collateral/funding logic.

So the repo is not merely a futures-stripping exercise. It is explicitly a collateral-aware pricing workflow.

## 4. How the Theory Maps into the Repo

The notebook is easier to follow if we tie each theoretical step to the implementation layer that performs it.

In [ ]:
implementation_map = pd.DataFrame(
    [
        ('Conventions and dates', 'src/daycount.py, src/calendars.py, src/dates.py', 'Year fractions, business-day adjustment, schedules, IMM logic'),
        ('Instrument layer', 'src/instruments.py', 'FuturesQuote, SwapQuote, coupon periods'),
        ('Curve representation', 'src/interpolation.py, src/curves.py', 'Log-discount-factor interpolation and forward extraction'),
        ('Hull-White convexity layer', 'src/hw_model.py', 'B(t,T), A(t,T), 1m convexity, 3m U_j term'),
        ('Bootstrap engine', 'src/bootstrap.py', 'OIS discount build, futures stripping, swap anchoring'),
        ('Pricing layer', 'src/pricers.py', 'Par swap rates and PV calculations'),
        ('Risk layer', 'src/risk.py', 'PV01, key-rate DV01, and scenario analysis'),
    ],
    columns=['Layer', 'Repo objects', 'Role in the notebook'],
)
implementation_map

## 5. Market Snapshot and Conventions

Before any curve math, we must pin down dates and accrual conventions. This matters because every futures and swap formula is sensitive to the accrual fraction $\tau$ or $\delta$.

The repo supports:

- `ACT/360`
- `ACT/365F`
- `30/360`
- `Following`
- `Modified Following`

In this demo, the synthetic market snapshot contains:

- historical SOFR fixings,
- 1m SOFR futures,
- 3m SOFR futures,
- SOFR swaps from `2Y` to `10Y`,
- OIS zero-rate pillars for collateral discounting.

In [ ]:
summary = pd.DataFrame(
    [
        ('Fixings', len(market.fixings), market.fixings['date'].min(), market.fixings['date'].max()),
        ('1m SOFR futures', len(market.futures_1m), market.futures_1m[0].start_date, market.futures_1m[-1].end_date),
        ('3m SOFR futures', len(market.futures_3m), market.futures_3m[0].start_date, market.futures_3m[-1].end_date),
        ('SOFR swaps', len(market.swaps), market.swaps[0].start_date, market.swaps[-1].end_date),
        ('OIS pillars', len(market.ois_curve), market.ois_curve['end_date'].min(), market.ois_curve['end_date'].max()),
    ],
    columns=['Dataset', 'Count', 'Start', 'End'],
)

display(summary)
display(market.fixings.tail())
display(market.ois_curve)
display(pd.DataFrame([vars(q) for q in market.futures_1m]))
display(pd.DataFrame([vars(q) for q in market.futures_3m]))
display(pd.DataFrame([vars(q) for q in market.swaps]))

## 6. Step A: Build the Collateral Discount Curve

The OIS discount curve is the collateral curve. At a given maturity $T$, a continuously compounded zero rate $z_d(T)$ corresponds to the discount factor

$$
P_d(0,T) = e^{-z_d(T)T}.
$$

Between market pillars, the repo interpolates on **log discount factors** rather than on raw rates. That means for an intermediate maturity $t$ between $T_i$ and $T_{i+1}$ we linearly interpolate

$$
\log P_d(0,t).
$$

This is a practical choice because it tends to preserve positive, monotone discount factors and gives more stable forward and swap repricing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(result.discount_curve.times[1:], result.discount_curve.dfs[1:], marker='o', label='OIS discount curve')
axes[0].plot(result.projection_curve.times[1:], result.projection_curve.dfs[1:], marker='s', label='SOFR projection curve')
axes[0].set_title('Discount Factors')
axes[0].set_xlabel('Years from valuation')
axes[0].set_ylabel('Discount factor')
axes[0].legend()

axes[1].plot(result.discount_curve.times[1:], result.discount_curve.pillar_zero_rates(), marker='o', label='OIS zeros')
axes[1].plot(result.projection_curve.times[1:], result.projection_curve.pillar_zero_rates(), marker='s', label='SOFR zeros')
axes[1].plot(result_no_convexity.projection_curve.times[1:], result_no_convexity.projection_curve.pillar_zero_rates(), linestyle='--', label='SOFR zeros, sigma = 0')
axes[1].set_title('Zero Curves')
axes[1].set_xlabel('Years from valuation')
axes[1].set_ylabel('Continuously compounded zero rate')
axes[1].legend()

plt.tight_layout()
plt.show()

pd.DataFrame({'Metric': list(result.diagnostics.keys()), 'Value': list(result.diagnostics.values())})

## 7. Step B: Strip the Short End from 1m SOFR Futures

Each 1m futures quote is first converted to an implied simple rate via

$$
R^{fut}_i = \frac{100 - \text{price}_i}{100}.
$$

Mercurio's setup adds a small convexity correction. In our constant-parameter implementation, the 1m convexity term is

$$
C^{1m}(T_s, T_e)
= \frac{\sigma^2}{2\delta a^2}
\left[
\delta
+ \frac{2}{a}e^{-aT_e}(1-e^{a\delta})
- \frac{1}{2a}e^{-2aT_e}(1-e^{2a\delta})
\right],
$$

with $\delta = T_e - T_s$ in year-fraction terms.

The code then strips the next SOFR pseudo-discount factor recursively as

$$
P_s(0,T_e) = P_s(0,T_s)\,e^{-\delta\,(R^{fut}_i - C^{1m}_i)}.
$$

This is implemented in `bootstrap_from_1m_futures(...)`.

In [ ]:
seed_dfs = bootstrap_sofr_short_end(valuation_date)
dfs_after_1m = bootstrap_from_1m_futures(market.futures_1m, valuation_date, a=mean_reversion, sigma=sigma, seed_dfs=seed_dfs)

one_m_steps = []
for quote in market.futures_1m:
    delta = yearfrac(quote.start_date, quote.end_date, 'ACT/360')
    t_start = yearfrac(valuation_date, quote.start_date, 'ACT/365F')
    t_end = yearfrac(valuation_date, quote.end_date, 'ACT/365F')
    adj = convexity_1m(mean_reversion, sigma, t_start, t_end)
    start_df = dfs_after_1m[quote.start_date]
    end_df = dfs_after_1m[quote.end_date]
    one_m_steps.append(
        {
            'Contract': quote.contract_code,
            'Start': quote.start_date,
            'End': quote.end_date,
            'Futures rate': quote.implied_rate,
            'Delta': delta,
            'Convexity adj (bp)': 10_000 * adj,
            'Start DF': start_df,
            'End DF': end_df,
        }
    )

pd.DataFrame(one_m_steps)

## 8. Step C: Extend the Curve with 3m SOFR Futures

For 3m futures, Mercurio's constant-parameter setup gives an exponential convexity correction $U_j$. The key relation is

$$
1 + \tau_j f^{3m,fut}_j(0)
= \frac{P_s(0,T_{j-1})}{P_s(0,T_j)} e^{U_j}.
$$

Therefore the SOFR pseudo-discount factor recursion becomes

$$
P_s(0,T_j) = \frac{P_s(0,T_{j-1})}{(1 + \tau_j f^{3m,fut}_j(0)) e^{U_j}}.
$$

In our implementation,

$$
U_j
= \frac{\sigma^2}{2a^3}\Big(
e^{-a(T_j + T_{j-1})}
- e^{-2aT_j}
+ e^{-a(T_j - T_{j-1})}
+ 2a(T_j - T_{j-1})
- 1
+ 2e^{-aT_j}
- 2e^{-aT_{j-1}}
\Big).
$$

This is implemented in `U_j_const_sigma(...)` and used by `bootstrap_from_3m_futures(...)`.

In [ ]:
dfs_after_3m = bootstrap_from_3m_futures(market.futures_3m, valuation_date, a=mean_reversion, sigma=sigma, seed_dfs=dfs_after_1m)

three_m_steps = []
for quote in market.futures_3m:
    tau = yearfrac(quote.start_date, quote.end_date, 'ACT/360')
    t_start = yearfrac(valuation_date, quote.start_date, 'ACT/365F')
    t_end = yearfrac(valuation_date, quote.end_date, 'ACT/365F')
    u_term = U_j_const_sigma(mean_reversion, sigma, t_start, t_end)
    start_df = dfs_after_3m[quote.start_date]
    end_df = dfs_after_3m[quote.end_date]
    three_m_steps.append(
        {
            'Contract': quote.contract_code,
            'Start': quote.start_date,
            'End': quote.end_date,
            'Futures rate': quote.implied_rate,
            'Tau': tau,
            'U_j (bp)': 10_000 * u_term,
            'Start DF': start_df,
            'End DF': end_df,
        }
    )

pd.DataFrame(three_m_steps)

## 9. Convexity Diagnostics and Repricing Check

Once the short and intermediate strip is built, the immediate validation question is whether the model reproduces the market futures. In this synthetic dataset, it should do so essentially exactly.

The convexity terms are small, but that is exactly the point: the project shows how to include them consistently rather than ignoring them.

In [ ]:
convexity_rows = []
for quote in market.futures_1m:
    t_start = yearfrac(valuation_date, quote.start_date, 'ACT/365F')
    t_end = yearfrac(valuation_date, quote.end_date, 'ACT/365F')
    convexity_rows.append(
        {
            'Instrument': quote.contract_code,
            'Type': 'SOFR_1M',
            'Maturity (y)': t_end,
            'Adjustment (bp)': 10_000 * convexity_1m(mean_reversion, sigma, t_start, t_end),
        }
    )
for quote in market.futures_3m:
    t_start = yearfrac(valuation_date, quote.start_date, 'ACT/365F')
    t_end = yearfrac(valuation_date, quote.end_date, 'ACT/365F')
    convexity_rows.append(
        {
            'Instrument': quote.contract_code,
            'Type': 'SOFR_3M',
            'Maturity (y)': t_end,
            'Adjustment (bp)': 10_000 * U_j_const_sigma(mean_reversion, sigma, t_start, t_end),
        }
    )
convexity_table = pd.DataFrame(convexity_rows)

display(convexity_table)
display(result.futures_repricing)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, group in convexity_table.groupby('Type'):
    axes[0].plot(group['Maturity (y)'], group['Adjustment (bp)'], marker='o', label=label)
axes[0].set_title('Convexity Terms by Maturity')
axes[0].set_xlabel('Years from valuation')
axes[0].set_ylabel('Adjustment (bp)')
axes[0].legend()

axes[1].bar(result.futures_repricing['Instrument'], result.futures_repricing['Error (bp)'])
axes[1].set_title('Futures Repricing Error (bp)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 10. Step D: Anchor the Medium and Long End with SOFR Swaps

Futures are not enough to build the full curve out to ten years, so we use SOFR swaps to anchor the medium and long end.

Given the SOFR pseudo-discount curve, the forward SOFR rate over period $[T_{i-1}, T_i]$ is

$$
F^s_i(0) = \frac{1}{\tau_i}\left(\frac{P_s(0,T_{i-1})}{P_s(0,T_i)} - 1\right).
$$

With collateral discount factors $P_d(0,T_i)$, the fixed-vs-SOFR par swap rate is

$$
S(0) = \frac{\sum_{i=1}^n \tau_i P_d(0,T_i) F^s_i(0)}{\sum_{i=1}^n \tau_i P_d(0,T_i)}.
$$

The bootstrap logic in the repo uses each quoted par swap rate to solve for the final unknown SOFR pseudo-discount factor at that maturity. That is exactly what turns a short-end futures strip into a usable long-end projection curve.

In [ ]:
display(result.swap_repricing)

plt.figure(figsize=(8, 4))
plt.bar(result.swap_repricing['Tenor'], result.swap_repricing['Error (bp)'])
plt.title('Swap Repricing Error (bp)')
plt.tight_layout()
plt.show()

## 11. Price a Collateral-Aware SOFR Swap

We now move from curve construction to valuation. For a notional $N$ and fixed rate $K$:

Fixed leg PV:

$$
PV_{fixed} = N \sum_{i=1}^n K\,\tau_i\,P_d(0,T_i).
$$

Floating leg PV:

$$
PV_{float} = N \sum_{i=1}^n \tau_i\,F^s_i(0)\,P_d(0,T_i).
$$

Swap PV:

$$
PV_{swap} = PV_{float} - PV_{fixed}.
$$

To make the model effect visible, we compare the same swap under:

- convexity on: the production Hull-White strip,
- convexity off: the same build with $\sigma = 0$.

In [ ]:
trade = market.swaps[-1]
periods = build_periods(
    trade.start_date,
    trade.end_date,
    trade.pay_freq,
    trade.day_count,
    calendar=market.config.market.calendar,
    roll=market.config.market.business_day_roll,
)

notional = 100_000_000
par_rate_on = par_swap_rate(result.discount_curve, result.projection_curve, periods)
par_rate_off = par_swap_rate(result_no_convexity.discount_curve, result_no_convexity.projection_curve, periods)
pv_on = pv_swap(notional, trade.fixed_rate, periods, result.discount_curve, result.projection_curve)
pv_off = pv_swap(notional, trade.fixed_rate, periods, result_no_convexity.discount_curve, result_no_convexity.projection_curve)

trade_summary = pd.DataFrame(
    [
        ('Market fixed rate', trade.fixed_rate),
        ('Model par rate, convexity on', par_rate_on),
        ('Model par rate, convexity off', par_rate_off),
        ('Swap PV, convexity on', pv_on),
        ('Swap PV, convexity off', pv_off),
        ('PV difference (on - off)', pv_on - pv_off),
    ],
    columns=['Metric', 'Value'],
)
trade_summary

## 12. Risk Layer: PV01, Key-Rate DV01, and Scenarios

The final step is to show that the project behaves like a small desk tool, not only a formula notebook.

The risk layer computes:

- PV01 from bump-and-reprice,
- key-rate DV01s,
- scenario PV and par rate shifts,
- a convexity-on versus convexity-off comparison,
- simple funding-spread and basis overlays.

The funding-spread scenarios are intentionally simple and pedagogical. They are not a full Piterbarg funding model, but they are included to make the economic message visible: changing the effective discounting environment changes PV.

In [ ]:
pv01_value = pv01(notional, trade.fixed_rate, periods, result.discount_curve, result.projection_curve)
krd = key_rate_dv01(
    notional,
    trade.fixed_rate,
    periods,
    result.discount_curve,
    result.projection_curve,
    list(market.config.risk.key_rates_years),
)

risk_table = pd.DataFrame(
    [('PV01', pv01_value)] + [(f'Key-rate DV01 {key:.0f}Y', value) for key, value in krd.items()],
    columns=['Risk Metric', 'Value'],
)
risk_table

In [ ]:
scenario_table = run_scenarios(
    notional,
    trade.fixed_rate,
    periods,
    result.discount_curve,
    result.projection_curve,
    alt_projection_curve=result_no_convexity.projection_curve,
)

display(scenario_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(scenario_table['Scenario'], scenario_table['Swap PV'])
axes[0].set_title('Scenario PV')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(scenario_table['Scenario'], scenario_table['Par Rate'])
axes[1].set_title('Scenario Par Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 13. What This Notebook Demonstrates Relative to the Papers

### Mercurio connection

This notebook mirrors the practical part of Mercurio's setup by:

- separating discounting from projection,
- expressing projected SOFR forwards through SOFR pseudo-discount factors,
- correcting futures-implied forwards with a Hull-White convexity layer,
- using swaps to complete the projection curve beyond the futures strip.

### Piterbarg connection

This notebook mirrors the core economic message in Piterbarg by:

- treating discounting as collateral-driven,
- separating discounting from forward generation,
- showing that funding/collateral assumptions can change PV and risk diagnostics.

### Important simplifications

This repo is an MVP, so it intentionally does **not** implement the full generality of either paper. In particular:

- basis is deterministic,
- the Hull-White factor uses constant $a$ and a smoothness-calibrated constant $\sigma$,
- prompt futures with realized-fixing splits are simplified away by the synthetic data,
- no stochastic basis, LMM, swaptions, or Monte Carlo is included.

## 14. References

1. Fabio Mercurio, *A Simple Multi-Curve Model for Pricing SOFR Futures and Other Derivatives* (2018 SSRN note), the source referenced locally as `sofr_mercurio_2.pdf`.
2. Vladimir Piterbarg, *Funding beyond discounting: collateral agreements and derivatives pricing* (2010), the source referenced locally as `2010_mar_funding piterbarg (2019_10_05 14_29_35 UTC).pdf`.

This notebook uses those references to motivate the model structure and valuation interpretation, while the numerical implementation is intentionally simplified for a readable, interview-friendly Python MVP.